In [ ]:
!git clone https://github.com/giambono/divine_semantics.git
import os
os.chdir("/content/divine_semantics")
!pip install -r requirements.txt

In [2]:
# from google.colab import drive
# drive.mount('/content/drive')

import pandas as pd
from src.find_similarity import find_most_similar_ensemble
from src.performance import evaluate_performance

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/387 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/160k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/57.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/690 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/418 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/280 [00:00<?, ?B/s]

1_Pooling/config.json:   0%|          | 0.00/201 [00:00<?, ?B/s]

In [3]:
from sentence_transformers import SentenceTransformer
df = pd.read_pickle("out/ensemble_embeddings.pkl")
df = df[['volume', 'canto', 'verse', 'dante', 'singleton', 'musa', 'kirkpatrick', 'durling']]

In [5]:
from src.compute import compute_ensemble_embeddings
models={"multilingual_e5": SentenceTransformer("intfloat/multilingual-e5-large")}

weights = {
    "dante": 0.0,
    "singleton": 0.1,
    "musa": 0.3,
    "kirkpatrick": 0.3,
    "durling": 0.3,
}

df = compute_ensemble_embeddings(df, columns=["singleton", "musa", "kirkpatrick", "durling"], models=models, weights=weights)
df.to_pickle(("out/ensemble_all_embeddings.pkl"))

Computing embeddings with multilingual_e5...


In [7]:
test_queries = pd.read_pickle("out/test_set.pkl")
test_queries = test_queries[["query", "expected_index"]]
test_queries = dict(zip(test_queries.iloc[:, 0], test_queries.iloc[:, 1]))

In [8]:
df, performance_results = evaluate_performance(df, models, test_queries)


Starting performance evaluation...
Evaluating multilingual_e5...


In [9]:
performance_results

{'multilingual_e5': {'embedding_multilingual_e5_singleton': 0.845213849287169,
  'embedding_multilingual_e5_musa': 0.7871690427698574,
  'embedding_multilingual_e5_kirkpatrick': 0.8054989816700611,
  'embedding_multilingual_e5_durling': 0.8503054989816701,
  'embedding_multilingual_e5_avg': 0.8762729124236253,
  'embedding_multilingual_e5_weighted': 0.8828920570264766}}